# 02 — EDA: Energy Grid Demand

Exploratory analysis of hourly grid demand (EIA, PJM balancing authority)
before moving to feature engineering. Mirrors `01_eda_ridership.ipynb`.

Run `ingestion/energy/fetch_eia_demand.py` first to produce the input file.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_parquet("../data/raw/energy/eia_demand_pjm.parquet")
df.head()

### 1. Date range and basic shape

In [ ]:
print(df["period"].min(), "to", df["period"].max())
print(df.shape)
df.dtypes

### 2. Missing values / reporting gaps

In [ ]:
df["value"].isna().sum(), df["value"].isna().mean()

### 3. Daily seasonality — hour-of-day pattern

In [ ]:
df["hour"] = df["period"].dt.hour
df.groupby("hour")["value"].mean().plot(title="Avg demand by hour of day")
plt.ylabel("MWh")
plt.show()

### 4. Weekly seasonality — day-of-week pattern

In [ ]:
df["dayofweek"] = df["period"].dt.dayofweek
df.groupby("dayofweek")["value"].mean().plot(kind="bar", title="Avg demand by day of week (0=Mon)")
plt.ylabel("MWh")
plt.show()

### 5. Yearly / seasonal pattern (this is where weather sensitivity should show up — summer AC + winter heating peaks)

In [ ]:
df["month"] = df["period"].dt.month
df.groupby("month")["value"].mean().plot(title="Avg demand by month")
plt.ylabel("MWh")
plt.show()

### 6. Full time series — look for anomalies (extreme weather events, outages, holidays)

In [ ]:
df.set_index("period")["value"].plot(figsize=(20, 5), title="Hourly PJM demand")
plt.ylabel("MWh")
plt.show()

### 7. Cross-check against Chicago Energy Benchmarking (spatial layer)

Load the building-level benchmarking data and look at electricity use
aggregated by `community_area` — this becomes the neighborhood weighting
applied to the hourly PJM series in `features/energy/`.

In [ ]:
bench = pd.read_parquet("../data/raw/energy/chicago_energy_benchmarking.parquet")
bench["electricity_use"] = pd.to_numeric(bench["electricity_use"], errors="coerce")
bench.groupby("community_area")["electricity_use"].sum().sort_values(ascending=False).head(15).plot(
    kind="barh", title="Total reported electricity use by community area (most recent year)"
)
plt.xlabel("kWh")
plt.show()

---
**Do not move to feature engineering until you can answer:**
- What does demand look like by hour, day, and month?
- Where are the anomalies (heat waves, cold snaps, outages)?
- Which community areas dominate reported building electricity use, and does that plausibly track with population/density?